# AI Health Copilot Pro — Phase 2: Exploratory Data Analysis (EDA)
**Author:** Dr. Harry Patria, Chief Data & AI Officer, Patria & Co.  
**CRISP-DM Phase:** Data Understanding & Clinical Phenotyping

This notebook audits the Pima Indians Diabetes dataset, checks the Outcome class imbalance, identifies zero-value data anomalies, and plots clinical distributions.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 300
os.makedirs("reports/eda", exist_ok=True)

# 1. Load Dataset
df = pd.read_csv("../dataset/diabetes.csv" if os.path.exists("../dataset/diabetes.csv") else "dataset/diabetes.csv")
print("Dataset Shape:", df.shape)
display(df.head())

In [2]:
# 2. Outcome Distribution (Countplot + Pie)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(data=df, x="Outcome", ax=axes[0], palette=["#0d9488", "#e11d48"])
axes[0].set_title("Outcome Frequency (0: Non-Diabetic, 1: Diabetic)")

counts = df["Outcome"].value_counts()
axes[1].pie(counts, labels=["Negative (0)", "Positive (1)"], autopct="%1.1f%%", colors=["#0d9488", "#e11d48"], startangle=90)
axes[1].set_title("Outcome Prevalence (~34.9% Class Imbalance)")
plt.tight_layout()
plt.savefig("reports/eda/01_outcome_distribution.png", dpi=300)
plt.show()

In [3]:
# 3. Audit Biologically Impossible Zeros
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for c in zero_cols:
    zeros = (df[c] == 0).sum()
    print(f"{c:15s}: {zeros:3d} zeros ({zeros / len(df) * 100:.1f}% missing)")

# Replace zeros with NaN
df_clean = df.copy()
for c in zero_cols:
    df_clean[c] = df_clean[c].replace(0, np.nan)

In [4]:
# 4. Correlation Heatmap
plt.figure(figsize=(10, 8))
corr = df_clean.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True, square=True)
plt.title("Biomarker Feature Correlation Matrix (Post-Zero Cleaned)")
plt.tight_layout()
plt.savefig("reports/eda/02_correlation_heatmap.png", dpi=300)
plt.show()

In [5]:
# 5. Boxplots: Glucose, BMI, Age, BloodPressure by Outcome
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.boxplot(data=df_clean, x="Outcome", y="Glucose", ax=axes[0, 0], palette=["#99f6e4", "#fecdd3"])
axes[0, 0].set_title("Fasting Glucose by Outcome")

sns.boxplot(data=df_clean, x="Outcome", y="BMI", ax=axes[0, 1], palette=["#99f6e4", "#fecdd3"])
axes[0, 1].set_title("Body Mass Index (BMI) by Outcome")

sns.boxplot(data=df_clean, x="Outcome", y="Age", ax=axes[1, 0], palette=["#99f6e4", "#fecdd3"])
axes[1, 0].set_title("Age by Outcome")

sns.boxplot(data=df_clean, x="Outcome", y="BloodPressure", ax=axes[1, 1], palette=["#99f6e4", "#fecdd3"])
axes[1, 1].set_title("Blood Pressure by Outcome")

plt.tight_layout()
plt.savefig("reports/eda/03_clinical_boxplots.png", dpi=300)
plt.show()